In [4]:
import pandas as pd
import numpy as np
from collections import Counter

In [11]:
data=pd.read_csv(r"../../data/processed/data_vif.csv")

data.drop("Date", axis=1, inplace=True)
data.replace({'Risk_Label': {'Low': 0, 'Medium': 1, 'High': 2}}, inplace=True)
data.head()

,KODEX 200_Premium,KOSDAQ_return(%),NASDAQ_return(%),Brent Crude Oil_return(%),Gold Spot_return(%),USD/KRW_change(%),KOSPI 200 Volume,return(%),KOSPI 200 lagged_1_return(%),KOSPI 200 lagged_2_return(%),...,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell,Risk_Label
0,-0.20,-2.796416,0.157320,0.0,-1.362586,-0.549095,251600000.0,-0.233192,0.303254,-0.477796,...,-2.736404,0,0,0,0,0,0,0,0,Low Risk
1,-0.34,1.668519,-3.953850,0.0,2.234472,0.128139,183300000.0,0.564563,-0.233192,0.303254,...,-2.640948,0,0,0,0,0,0,0,0,Low Risk
2,-0.14,1.061549,2.191930,0.0,-0.553958,1.998728,204200000.0,-0.197523,0.564563,-0.233192,...,-2.553087,0,0,0,0,0,0,0,0,Low Risk
3,-0.19,2.524236,0.137996,0.0,1.093648,-0.570187,291400000.0,1.408955,-0.197523,0.564563,...,-2.469280,0,0,0,0,0,0,0,0,Low Risk
4,-0.17,0.818378,0.369276,0.0,1.568707,-0.970084,277400000.0,0.992765,1.408955,-0.197523,...,-2.386329,0,0,0,0,0,1,0,0,Low Risk


In [ ]:
step=['NASDAQ_return(%)',
     'KOSPI 200_BB_width',
     'KOSPI 200_PPO_Hist',
     'KOSPI 200_OG',
     'KODEX 200_Premium',
     'Signal2_Buy', 'Risk_Label']

mul=['NASDAQ_return(%)', 'Gold Spot_return(%)', 
     'Brent Crude Oil_return(%)', 'KOSPI 200 lagged_1_return(%)', 
     'VKOSPI_Close', 'return(%)', 'KOSPI 200_ADX14',
     'VKOSPI_Change(%)', 'KOSDAQ_return(%)', 'KOSPI 200_PPO',
     'USD/KRW_change(%)', 'KOSPI 200_BB_width', 'Signal2_Buy', 
     'Signal2_Sell', 'Risk_Label']

boruta=['KOSPI 200_PPO_Hist', 'KOSPI 200 Volume', 
        'KOSPI 200_OG', 'KOSPI 200_DMI14', 
        'GJR_VaR_5_t1', 'Brent Crude Oil_return(%)',
        'VKOSPI_Close', 'NASDAQ_return(%)', 'Risk_Label']


In [13]:

# ============================================================
# #1. 하드보팅 함수
# ============================================================

def hard_voting_feature_selection(
    step_features,
    mul_features,
    boruta_features,
    target_col='Risk_Label',
    vote_threshold=2
):

    selection_dict = {
        'step': step_features,
        'mul': mul_features,
        'boruta': boruta_features
    }

    # 타겟 제거 + 중복 제거
    cleaned = {}
    for method, features in selection_dict.items():
        cleaned[method] = list(dict.fromkeys([
            f for f in features 
            if f != target_col
        ]))

    # 전체 변수 union
    all_features = sorted(set().union(*cleaned.values()))

    # 투표 테이블 생성
    rows = []
    for feature in all_features:
        selected_by = []
        vote_count = 0

        for method, features in cleaned.items():
            is_selected = feature in features
            if is_selected:
                vote_count += 1
                selected_by.append(method)

        rows.append({
            'feature': feature,
            'vote_count': vote_count,
            'selected_by_step': feature in cleaned['step'],
            'selected_by_mul': feature in cleaned['mul'],
            'selected_by_boruta': feature in cleaned['boruta'],
            'selected_by': ', '.join(selected_by)
        })

    vote_table = pd.DataFrame(rows)

    # 보기 좋게 정렬
    vote_table = vote_table.sort_values(
        by=['vote_count', 'feature'],
        ascending=[False, True]
    ).reset_index(drop=True)

    # 최종 선택 변수
    selected_features = vote_table.loc[
        vote_table['vote_count'] >= vote_threshold,
        'feature'
    ].tolist()

    return selected_features, vote_table


# ============================================================
# #2. 보팅 실행
# ============================================================

selected_features_vote, vote_table = hard_voting_feature_selection(
    step_features=step,
    mul_features=mul,
    boruta_features=boruta,
    target_col='Risk_Label',
    vote_threshold=2
)

print("=" * 80)
print("최종 선택 변수 개수:", len(selected_features_vote))
print("=" * 80)

for i, col in enumerate(selected_features_vote, 1):
    print(f"{i}. {col}")

print("\n" + "=" * 80)
print("전체 투표 결과")
print("=" * 80)

display(vote_table)


# ============================================================
# #3. 최종 모델링용 컬럼 생성
# ============================================================

final_cols = selected_features_vote + ['Risk_Label']

print("\n최종 사용 컬럼:")
print(final_cols)

최종 선택 변수 개수: 7
1. NASDAQ_return(%)
2. Brent Crude Oil_return(%)
3. KOSPI 200_BB_width
4. KOSPI 200_OG
5. KOSPI 200_PPO_Hist
6. Signal2_Buy
7. VKOSPI_Close

전체 투표 결과


,feature,vote_count,selected_by_step,selected_by_mul,selected_by_boruta,selected_by
0,NASDAQ_return(%),3,True,True,True,"step, mul, boruta"
1,Brent Crude Oil_return(%),2,False,True,True,"mul, boruta"
2,KOSPI 200_BB_width,2,True,True,False,"step, mul"
3,KOSPI 200_OG,2,True,False,True,"step, boruta"
4,KOSPI 200_PPO_Hist,2,True,False,True,"step, boruta"
5,Signal2_Buy,2,True,True,False,"step, mul"
6,VKOSPI_Close,2,False,True,True,"mul, boruta"
7,GJR_VaR_5_t1,1,False,False,True,boruta
8,Gold Spot_return(%),1,False,True,False,mul
9,KODEX 200_Premium,1,True,False,False,step



최종 사용 컬럼:
['NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_OG', 'KOSPI 200_PPO_Hist', 'Signal2_Buy', 'VKOSPI_Close', 'Risk_Label']


In [14]:
data[final_cols].to_csv(r"../../data/processed/data_selected.csv", index=False)